# ResearchLM

## Load documents

In [5]:
import os
import glob
import numpy as np
import tiktoken
from dotenv import load_dotenv
from IPython.display import display, Markdown

In [6]:
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.manifold import TSNE

In [7]:
MODEL = "llama-3.1-8b-instant"

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:8]}")
else:
    print("GROQ API Key not set")


GROQ API Key exists and begins gsk_tGmX


In [8]:
def get_pdf_files(
    knowledge_base_path="./data/**/*.pdf"
):

    files = glob.glob(
        knowledge_base_path,
        recursive=True
    )

    print(f"Found {len(files)} PDF files")

    return files

In [9]:
pdf_files = get_pdf_files()

Found 4 PDF files


In [10]:
def count_total_characters(pdf_files):

    total_characters = 0

    for file_path in pdf_files:

        loader = PyPDFLoader(file_path)
        pages = loader.load()

        for page in pages:
            total_characters += len(page.page_content)

    print(
        f"Total characters in knowledge base: "
        f"{total_characters:,}"
    )

    return total_characters

In [11]:
count_total_characters(pdf_files)

Total characters in knowledge base: 202,241


202241

In [12]:
import os
import glob
from tqdm.auto import tqdm

def load_documents(knowledge_base_path="./data/**/*.pdf"):
    pdf_files = glob.glob(knowledge_base_path, recursive=True)
    documents = []

    # Add tqdm around the pdf_files list
    for file_path in tqdm(pdf_files, desc="Parsing PDF Files"):
        loader = PyPDFLoader(file_path)
        docs = loader.load()

        for doc in docs:
            doc.metadata["doc_type"] = "research_paper"
            doc.metadata["file_name"] = os.path.basename(file_path)
            documents.append(doc)

    print(f"Loaded {len(documents)} pages")
    return documents

# Execute the updated loader
raw_documents = load_documents()


Parsing PDF Files: 100%|██████████| 4/4 [00:05<00:00,  1.50s/it]

Loaded 53 pages


In [13]:
raw_documents[0]

Document(metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2016-05-23T00:19:15+00:00', 'author': '', 'keywords': '', 'moddate': '2016-05-23T00:19:15+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data\\1409.0473v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'doc_type': 'research_paper', 'file_name': '1409.0473v7.pdf'}, page_content='Published as a conference paper at ICLR 2015\nNEURAL MACHINE TRANSLATION\nBY JOINTLY LEARNING TO ALIGN AND TRANSLATE\nDzmitry Bahdanau\nJacobs University Bremen, Germany\nKyungHyun Cho Y oshua Bengio ∗\nUniversit´e de Montr´eal\nABSTRACT\nNeural machine translation is a recently proposed approach to machine transla-\ntion. Unlike the traditional statistical machine translation, the neural machine\ntranslation aims at building a single neural network that can be jointly t

## Phase 1 - Parent-Child Chunking

In [14]:
import os
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

# 1. Initialize your embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Configure the dual-layer splitters
# Parent chunks provide the broad context to the LLM
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

# Child chunks provide high-precision semantic matching in the vector space
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

# 3. Set up the storage layers
# The vectorstore holds the small embedded child chunks
db_name = "advanced_vector_db"
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma(
    collection_name="split_parents", 
    embedding_function=embeddings, 
    persist_directory=db_name
)

# The InMemoryStore holds the large raw text parent chunks
store = InMemoryStore()

# 4. Initialize the advanced retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 5. Ingest the documents with tqdm batching
batch_size = 5 # Adjust based on your system RAM; 5 pages at a time is safe

# Calculate total batches for tqdm
total_batches = (len(raw_documents) + batch_size - 1) // batch_size

for i in tqdm(range(0, len(raw_documents), batch_size), desc="Ingesting to Vector Database"):
    batch = raw_documents[i : i + batch_size]
    retriever.add_documents(batch)

print(f"Successfully processed {len(raw_documents)} raw document pages into hierarchical chunks.")

Ingesting to Vector Database: 100%|██████████| 11/11 [00:10<00:00,  1.09it/s]

Successfully processed 53 raw document pages into hierarchical chunks.


In [15]:
# Run a test query
test_query = "How the Navigable Small World works?"
retrieved_docs = retriever.invoke(test_query)

# Check the length of the retrieved context. 
# It should be close to your parent_splitter chunk_size (~2000 characters), 
# proving it pulled the parent, not just the 400-character child.
print(f"Retrieved {len(retrieved_docs)} parent chunks.")
print(f"Length of top chunk: {len(retrieved_docs[0].page_content)} characters.")

Retrieved 4 parent chunks.
Length of top chunk: 1933 characters.


In [17]:
print(retrieved_docs[1])

page_content='IEEE TRANSACTIONS ON JOURNAL NAME,  MANUSCRIPT ID 1 
 
Efficient and robust approximate nearest 
neighbor search using Hierarchical Navigable 
Small World graphs  
Yu. A. Malkov, D. A. Yashunin 
Abstract — We present a new approach for the approximate K -nearest neighbor search based on navigable small world 
graphs with controllable hierarchy (Hierarchical NSW, HNSW). The proposed solution is fully graph-based, without any need for 
additional search structures, which are typically used at the coarse search stage of the most proximity graph techniques. 
Hierarchical NSW incrementally builds a multi-layer structure consisting from hierarchical set of proximity graphs (layers) for 
nested subsets of the stored elements. The maximum layer in which an element is present is selected randomly with an 
exponentially decaying probability distribution. This allows producing graphs similar to the previously studied Navigable Sma ll 
World (NSW) struc tures while additionally havin

## Phase 2 - Hybrid Search (BM25 + Dense) + Re-ranker

In [18]:
# pip install rank_bm25 sentence-transformers
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# --- Step 1: Initialize the Sparse Retriever (Keyword Search) ---
# We build the BM25 retriever using the raw_documents loaded in Phase 1
bm25_retriever = BM25Retriever.from_documents(raw_documents)
bm25_retriever.k = 5 # Retrieve the top 5 keyword matches

# --- Step 2: Prepare the Dense Retriever (Semantic Search) ---
# We use the 'retriever' we built in Phase 1 (your ParentDocumentRetriever)
dense_retriever = retriever 
dense_retriever.search_kwargs = {"k": 5} # Retrieve the top 5 semantic matches

# --- Step 3: Combine them using Hybrid Search (Ensemble) ---
# The EnsembleRetriever uses RRF to fuse the ranked lists.
# Weights determine the bias. [0.5, 0.5] means equal priority.
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5]
)

# --- Step 4: Initialize the Cross-Encoder Re-ranker ---
# BAAI/bge-reranker-base is highly optimized for English retrieval tasks.
# cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# Set top_n=3 to ensure only the 3 most strictly relevant chunks are sent to the LLM.
# This saves context window tokens and prevents the LLM from getting distracted.
compressor = CrossEncoderReranker(model=cross_encoder, top_n=3)

# --- Step 5: Wrap the pipeline into the final Advanced Retriever ---
advanced_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever
)

print("Phase 2: Hybrid Search + Re-ranker successfully initialized!")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11312.08it/s]


Phase 2: Hybrid Search + Re-ranker successfully initialized!


In [19]:
# Test the advanced retriever
test_query = "How the Navigable Small World works?"
compressed_docs = advanced_retriever.invoke(test_query)

print(f"Final chunks passed to LLM: {len(compressed_docs)}")
for i, doc in enumerate(compressed_docs):
    print(f"\n--- Rank {i+1} ---")
    print(f"Content: {doc.page_content[:500]}")
    print(f"Length: {len(doc.page_content)}")
    print(f"Relevance Score: {doc.metadata.get('relevance_score', 'N/A')}")
    print(f"Source: {doc.metadata.get('file_name')}")

Final chunks passed to LLM: 3

--- Rank 1 ---
Content: the greedy graph r outing are known as the navigable 
small world networks  [31, 32] . Such networks are an i m-
portant topic of complex network theory aiming at u n-
derstanding of underlying mechanisms of real-life ne t-
works formation in order to apply them for appli cations 
of scalable routing [32, 35, 36]  and distributed similarity 
search [25, 26, 30, 37-40].  
The first works to consider spatial models of navigable 
networks were done by J.  Kleinberg [31, 41] as social ne t-
work mod
Length: 1955
Relevance Score: N/A
Source: 1603.09320v4.pdf

--- Rank 2 ---
Content: that use auxiliary algorithms applicable only for vector 
data (such as kd -trees [18, 19] and product 
quantization [10]) to find better candidates for the enter 
nodes by doing a coarse search.  
In [25, 26, 30]  authors proposed a proximity graph 
K-ANNS algorithm called Navigable Small World (NSW, 
also known as Metricized Small World, MSW), which 
utili

## LLM Generation

In [20]:
llm = ChatGroq(temperature=0.7, model_name=MODEL)

In [21]:
SYSTEM_PROMPT_TEMPLATE = """
You are an expert AI research assistant specializing in synthesizing and analyzing academic literature.

Your sole task is to provide technically precise, comprehensive, and objective answers to the user's question using ONLY the provided context blocks. 

---

### STRICT OPERATIONAL DIRECTIVES:
1. **Zero External Knowledge:** Rely entirely on the provided context. Never introduce outside facts, industry assumptions, or extrapolated theories.
2. **Anti-Hallucination Guardrail:** If the exact answer cannot be fully constructed using the provided context, state exactly: "Not enough information available in the provided documents." Do not attempt to bridge gaps with speculation.
3. **Seamless In-Text Citations:** Every claim, fact, or metric you state MUST be immediately followed by an inline citation referencing its source file name and page/section number if available in the context metadata.
4. **Contradiction Resolution:** If different papers provide conflicting information on the same topic, explicitly highlight the variance (e.g., "Paper A states X, whereas Paper B finds Y").
5. **No Meta-Language:** Do not use conversational filler or phrases that reference the system architecture, such as "Based on the context provided," "According to the text," or "The documents show." Jump directly into the factual analysis.

---

### CITATION FORMAT EXAMPLE:
"The Transformer architecture replaces recurrent layers entirely with stacked self-attention mechanisms, significantly reducing training times [Source: Attention_Is_All_You_Need.pdf, Page: 3]. However, localized context retention can still vary depending on positional encoding choices [Source: Transformer_Analysis.pdf, Page: 7]."

---

### CONTEXT:
{context}

### QUESTION:
{question}

### ANSWER:
"""

In [22]:
def answer_question(question: str, history=None):

    # Retrieve docs using the new Hybrid + Reranker pipeline
    docs = advanced_retriever.invoke(question)

    # Build context with metadata
    context_parts = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get(
            "file_name",
            "Unknown Source"
        )
        page = doc.metadata.get(
            "page",
            "Unknown Page"
        )
        chunk_text = f"""
        Document [{i}]
        Source: {source}
        Page: {page}

        Content:
        {doc.page_content}
        """
        context_parts.append(chunk_text)
    context = "\n\n".join(context_parts)
    
    # Create prompt
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context, question=question)
    
    # Generate response
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])

    return response.content

In [23]:
response = answer_question("How the Navigable Small World works?")

In [24]:
display(Markdown(response))

The Navigable Small World (NSW) graph is constructed via consecutive insertion of elements in random order by bidirectionally connecting them to the M closest neighbors from the previously inserted elements [Source: 1603.09320v4.pdf, Page: 1]. The M closest neighbors are found using the structure's search procedure (a variant of a greedy search from multiple random enter nodes) [Source: 1603.09320v4.pdf, Page: 1]. Links to the closest neighbors of the elements inserted in the beginning of the construction later become bridges between the network hubs that keep the overall graph connectivity and allow the logarithmic scaling of the number of hops during greedy routing [Source: 1603.09320v4.pdf, Page: 1].

## Phase 3 - Production-Level Evaluation using Ragas.

In [36]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness, 
    answer_relevancy, 
    context_precision, 
    context_recall
)

# --- Step 1: Define your Evaluation Dataset ---
# In a real production system, you'd pull this from a CSV of past user queries.
eval_questions = [
    "What is the methodology used in YOLO?",
    "What is the architecture of the Transformer model?"
]

# Ground truths are required for 'context_recall' (did we find everything we needed?)
eval_ground_truths = [
    "YOLO frames object detection as a single regression problem, straight from image pixels to bounding box coordinates and class probabilities.",
    "The Transformer relies entirely on an attention mechanism, dispensing with recurrence and convolutions, using stacked self-attention and point-wise, fully connected layers."
]

answers = []
contexts = []

# --- Step 2: Run the Pipeline to Gather Data ---
print("Running pipeline to gather generation and retrieval data...")

for q in eval_questions:
    # 1. Get the exact chunks the retriever pulled
    retrieved_docs = advanced_retriever.invoke(q)
    context_list = [doc.page_content for doc in retrieved_docs]
    contexts.append(context_list)
    
    # 2. Get the final LLM generation
    # Make sure your answer_question function is using advanced_retriever!
    llm_answer = answer_question(q) 
    answers.append(llm_answer)

# --- Step 3: Format Data for Ragas ---
data = {
    "user_input": eval_questions,    
    "response": answers,             
    "retrieved_contexts": contexts,  
    "reference": eval_ground_truths  
}
dataset = Dataset.from_dict(data)

# --- Step 4: Run the Evaluation (Updated for Groq & HuggingFace) ---
print("Evaluating pipeline performance using Ragas...")
result = evaluate(
    dataset,
    metrics=[
        faithfulness,      
        answer_relevancy,  
        context_precision, 
        context_recall     
    ],
    llm=llm,              
    embeddings=embeddings 
)

# Convert to a clean Pandas DataFrame for easy viewing
df_results = result.to_pandas()
display(df_results)

C:\Users\admin\AppData\Local\Temp\ipykernel_2616\1797848271.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\admin\AppData\Local\Temp\ipykernel_2616\1797848271.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\admin\AppData\Local\Temp\ipykernel_2616\1797848271.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\admin\AppData\Local\Temp\ipykernel_2616\

Running pipeline to gather generation and retrieval data...
Evaluating pipeline performance using Ragas...


Evaluating: 100%|██████████| 8/8 [03:00<00:00, 22.50s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the methodology used in YOLO?,[making predictions. Unlike sliding window and...,YOLO (You Only Look Once) sees the entire imag...,YOLO frames object detection as a single regre...,NaN,NaN,NaN,NaN
1,What is the architecture of the Transformer mo...,[Figure 1: The Transformer - model architectur...,The Transformer model follows a stacked archit...,The Transformer relies entirely on an attentio...,NaN,NaN,NaN,NaN
